# BBBC021 morphology classification: build a patch dataset

This notebook prepares a balanced five-class image dataset from BBBC021 for classification and Grad-CAM. It supports three dataset scales selected through an enum:

- `SMOKE_TEST`: pipeline verification in a few minutes
- `DEVELOPMENT`: practical default for standard Colab storage
- `FULL`: 200,000-patch dataset

The biological hierarchy is `compound → concentration → plate → well → field of view → patch`. Splits are assigned at plate-well level before patch extraction.

## 1. Colab setup

The repository is cloned and installed from its root so project-specific code is importable without modifying `sys.path`.

In [ ]:
!git clone -q https://github.com/lohex/DeeplearningExamples.git
%cd DeeplearningExamples
%pip install -q -e ".[bbbc021]" 

## 2. Dataset-scale presets

In [ ]:
from dataclasses import asdict, dataclass
from enum import Enum
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from bbbc021_morphology_classification.pipeline import (
    PLATE,
    PatchExtractionConfig,
    allocate_patches,
    assign_splits,
    create_paths,
    download_source_images,
    generate_dataset,
    load_selected_metadata,
    pilot_storage_estimate,
    source_summary,
    validate_manifest,
)


class DatasetScale(str, Enum):
    SMOKE_TEST = "smoke_test"
    DEVELOPMENT = "development"
    FULL = "full"


@dataclass(frozen=True, slots=True)
class ScaleConfig:
    total_patches: int
    shard_size: int
    max_plates_per_class: int | None
    max_wells_per_class: int | None


SCALE_CONFIGS = {
    DatasetScale.SMOKE_TEST: ScaleConfig(
        total_patches=1_000,
        shard_size=500,
        max_plates_per_class=1,
        max_wells_per_class=3,
    ),
    DatasetScale.DEVELOPMENT: ScaleConfig(
        total_patches=10_000,
        shard_size=1_000,
        max_plates_per_class=2,
        max_wells_per_class=10,
    ),
    DatasetScale.FULL: ScaleConfig(
        total_patches=200_000,
        shard_size=2_000,
        max_plates_per_class=None,
        max_wells_per_class=None,
    ),
}

DATASET_SCALE = DatasetScale.DEVELOPMENT
scale_config = SCALE_CONFIGS[DATASET_SCALE]

config = PatchExtractionConfig(
    work_dir=Path("/content/bbbc021_morphology"),
    total_patches=scale_config.total_patches,
    shard_size=scale_config.shard_size,
)
paths = create_paths(config)

asdict(scale_config), asdict(config)

The development preset generates 10,000 patches from at most two plates and ten wells per class. This reduces both output size and source-archive downloads. The full preset retains every available source well.

## 3. Select plates and wells

In [ ]:
def limit_source_groups(
    metadata: pd.DataFrame,
    max_plates_per_class: int | None,
    max_wells_per_class: int | None,
    seed: int,
) -> pd.DataFrame:
    selected_classes: list[pd.DataFrame] = []

    for _, class_data in metadata.groupby("class_name", sort=True):
        selected = class_data.copy()

        if max_plates_per_class is not None:
            plate_counts = selected.groupby(PLATE).size().sort_values(ascending=False)
            selected_plates = plate_counts.head(max_plates_per_class).index
            selected = selected[selected[PLATE].isin(selected_plates)]

        if max_wells_per_class is not None:
            well_ids = selected["group_id"].drop_duplicates()
            selected_wells = well_ids.sample(
                n=min(max_wells_per_class, len(well_ids)),
                random_state=seed,
            )
            selected = selected[selected["group_id"].isin(selected_wells)]

        selected_classes.append(selected)

    return pd.concat(selected_classes, ignore_index=True)


metadata = load_selected_metadata(config, paths)
metadata = limit_source_groups(
    metadata=metadata,
    max_plates_per_class=scale_config.max_plates_per_class,
    max_wells_per_class=scale_config.max_wells_per_class,
    seed=config.seed,
)
metadata = assign_splits(metadata, config)

source_summary(metadata)

BBBC021 is a fixed 24-hour endpoint assay, not a time series. One metadata row represents one FOV with DAPI, actin and tubulin channels.

## 4. Download the required plate archives

In [ ]:
archive_manifest = download_source_images(metadata, config, paths)
display(archive_manifest)
print(f"Required source download: {archive_manifest['GiB'].sum():.2f} GiB")

## 5. Estimate output size from pilot patches

In [ ]:
storage_estimate = pilot_storage_estimate(metadata, config, paths)
storage_estimate

## 6. Allocate and generate balanced patches

In [ ]:
allocation = allocate_patches(config)
allocation.pivot(index="class_name", columns="split", values="patches")

In [ ]:
patch_manifest = generate_dataset(
    metadata=metadata,
    allocation=allocation,
    config=config,
    paths=paths,
)
print(f"Wrote {len(patch_manifest):,} patches to {paths.output}")

## 7. Validate splits and inspect effective sample structure

In [ ]:
summary = validate_manifest(patch_manifest, config)
display(summary)

axis = (
    patch_manifest.groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .plot(kind="bar", stacked=True, figsize=(10, 5))
)
axis.set_ylabel("Patches")
axis.set_title(f"Patch allocation: {DATASET_SCALE.value}")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

The number of patches is not the number of independent biological observations. Report the number of source FOVs, wells and plates, and aggregate downstream test predictions at least by FOV or well.